# 00 — Environment Setup & Verification
**Drug Discovery Pipeline | RTX 3070 Optimized**

Run this once before any other notebook. If all cells pass, your environment is ready.

## 1. Install Dependencies

In [5]:
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

packages = [
    'numpy', 'pandas', 'scikit-learn', 'scipy',
    'rdkit', 'PyTDC',
    'torch-geometric',
    'deepchem',
    'transformers', 'datasets', 'tokenizers',
    'codecarbon', 'psutil', 'gpustat', 'pynvml',
    'optuna',
    'matplotlib', 'seaborn', 'plotly', 'umap-learn',
    'tqdm', 'ipywidgets', 'joblib',
]

print('Installing packages...')
for pkg in packages:
    try:
        install(pkg)
        print(f'  ✓ {pkg}')
    except Exception as e:
        print(f'  ✗ {pkg} — {e}')
print('Done.')

Installing packages...
  ✓ numpy
  ✓ pandas
  ✓ scikit-learn
  ✓ scipy
  ✓ rdkit
  ✗ PyTDC — Command '['C:\\Users\\PC\\anaconda3\\envs\\drug_discovery\\python.exe', '-m', 'pip', 'install', '-q', 'PyTDC']' returned non-zero exit status 1.
  ✓ torch-geometric
  ✓ deepchem
  ✓ transformers
  ✓ datasets
  ✓ tokenizers
  ✓ codecarbon
  ✓ psutil
  ✓ gpustat
  ✓ pynvml
  ✓ optuna
  ✓ matplotlib
  ✓ seaborn
  ✓ plotly
  ✓ umap-learn
  ✓ tqdm
  ✓ ipywidgets
  ✓ joblib
Done.


## 2. GPU Verification

In [6]:
import torch

print('='*55)
print('GPU VERIFICATION')
print('='*55)

if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name   = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_vram  = (torch.cuda.get_device_properties(0).total_memory
                  - torch.cuda.memory_allocated(0)) / 1e9
    print(f'  GPU:        {gpu_name}')
    print(f'  VRAM Total: {total_vram:.1f} GB')
    print(f'  VRAM Free:  {free_vram:.1f} GB')
    print(f'  CUDA:       {torch.version.cuda}')
    print(f'  PyTorch:    {torch.__version__}')
    if total_vram < 7.5:
        print('  ⚠ WARNING: Expected ~8GB for RTX 3070')
    else:
        print('  ✓ RTX 3070 confirmed — 8GB VRAM budget ready')
        print('  ✓ Mixed precision (FP16) will be enabled by default')
else:
    device = torch.device('cpu')
    print('  ✗ CUDA not available — running on CPU')
    print('  Install CUDA PyTorch:')
    print('  pip install torch --index-url https://download.pytorch.org/whl/cu121')

print('='*55)
print(f'  Active device: {device}')
print('='*55)

GPU VERIFICATION
  GPU:        NVIDIA GeForce RTX 3080
  VRAM Total: 12.9 GB
  VRAM Free:  12.9 GB
  CUDA:       12.1
  PyTorch:    2.5.1+cu121
  ✓ RTX 3070 confirmed — 8GB VRAM budget ready
  ✓ Mixed precision (FP16) will be enabled by default
  Active device: cuda


## 3. Memory Configuration

In [7]:
import torch, os

USE_AMP = torch.cuda.is_available()
scaler  = torch.cuda.amp.GradScaler(enabled=USE_AMP)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.backends.cudnn.benchmark = True

print('Mixed Precision (AMP):    ', '✓ ENABLED' if USE_AMP else '✗ disabled (CPU mode)')
print('Memory Allocator:          ✓ expandable_segments enabled')
print('cuDNN Benchmark Mode:      ✓ enabled')
print()
print('Estimated VRAM per model (with AMP):')
vram_budget = {
    'RF + Morgan FP':  '< 0.1 GB  (CPU — no GPU needed)',
    'AttentiveFP':     '~ 1.2 GB  (safe)',
    'MPNN (DeepChem)': '~ 2.5 GB  (safe)',
    'ChemBERTa-2':     '~ 5.0 GB  (safe with AMP)',
}
for model, budget in vram_budget.items():
    print(f'  {model:<20} {budget}')

GPU_CONFIG = {
    'device':      device,
    'use_amp':     USE_AMP,
    'scaler':      scaler,
    'batch_size':  64,
    'num_workers': 4,
    'pin_memory':  True,
}
print('\nGPU config ready.')

Mixed Precision (AMP):     ✓ ENABLED
Memory Allocator:          ✓ expandable_segments enabled
cuDNN Benchmark Mode:      ✓ enabled

Estimated VRAM per model (with AMP):
  RF + Morgan FP       < 0.1 GB  (CPU — no GPU needed)
  AttentiveFP          ~ 1.2 GB  (safe)
  MPNN (DeepChem)      ~ 2.5 GB  (safe)
  ChemBERTa-2          ~ 5.0 GB  (safe with AMP)

GPU config ready.


C:\Users\PC\AppData\Local\Temp\ipykernel_37856\4079941741.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## 4. CodeCarbon — Energy Tracking Setup

In [8]:
from codecarbon import EmissionsTracker
import os, torch

os.makedirs('./results', exist_ok=True)

# country_iso_code was removed in newer codecarbon versions — auto-detects location instead
CARBON_CONFIG = {
    'project_name': 'drug_discovery_benchmark',
    'output_dir':   './results/',
    'output_file':  'emissions.csv',
    'save_to_file': True,
    'log_level':    'warning',
}

print('Testing CodeCarbon tracker...')
tracker = EmissionsTracker(**CARBON_CONFIG)
tracker.start()

# Dummy GPU workload
if torch.cuda.is_available():
    x = torch.randn(1000, 1000, device='cuda')
    _ = x @ x
    del x
    torch.cuda.empty_cache()

emissions = tracker.stop()

print(f'  ✓ Tracker working')
print(f'  Emissions from test run: {emissions * 1000:.4f} mg CO2')
print(f'  Results saved to: ./results/emissions.csv')
print()
print('Usage in other notebooks:')
print('  tracker = EmissionsTracker(**CARBON_CONFIG)')
print('  tracker.start()')
print('  # ... train model ...')
print('  emissions = tracker.stop()  # returns kg CO2')

[codecarbon WARNING @ 12:37:51] Multiple instances of codecarbon are allowed to run at the same time.


Testing CodeCarbon tracker...


[codecarbon WARNING @ 12:37:54] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Windows OS detected: Please install Intel Power Gadget to measure CPU

[codecarbon WARNING @ 12:37:54] No CPU tracking mode found. Falling back on CPU load mode.


  ✓ Tracker working
  Emissions from test run: 0.0092 mg CO2
  Results saved to: ./results/emissions.csv

Usage in other notebooks:
  tracker = EmissionsTracker(**CARBON_CONFIG)
  tracker.start()
  # ... train model ...
  emissions = tracker.stop()  # returns kg CO2


## 5. Dependency Smoke Tests

In [9]:
results = {}

# RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, AllChem
    mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
    fp  = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    mw  = Descriptors.MolWt(mol)
    results['RDKit'] = f'✓  Aspirin MW={mw:.1f}, FP len={len(fp)}'
except Exception as e:
    results['RDKit'] = f'✗  {e}'

# PyTDC
try:
    from tdc.single_pred import ADME
    results['PyTDC'] = '✓  Imported successfully'
except Exception as e:
    results['PyTDC'] = f'✗  {e}'

# DeepChem
try:
    import deepchem as dc
    results['DeepChem'] = f'✓  v{dc.__version__}'
except Exception as e:
    results['DeepChem'] = f'✗  {e}'

# Transformers
try:
    import transformers
    results['Transformers'] = f'✓  v{transformers.__version__}'
except Exception as e:
    results['Transformers'] = f'✗  {e}'

# PyTorch Geometric
try:
    import torch_geometric
    results['PyTorch Geometric'] = f'✓  v{torch_geometric.__version__}'
except Exception as e:
    results['PyTorch Geometric'] = f'✗  {e}'

# Optuna
try:
    import optuna
    results['Optuna'] = f'✓  v{optuna.__version__}'
except Exception as e:
    results['Optuna'] = f'✗  {e}'

# UMAP
try:
    import umap
    results['UMAP'] = '✓  Imported successfully'
except Exception as e:
    results['UMAP'] = f'✗  {e}'

# CodeCarbon
try:
    from codecarbon import EmissionsTracker
    results['CodeCarbon'] = '✓  Imported successfully'
except Exception as e:
    results['CodeCarbon'] = f'✗  {e}'

print('='*55)
print('SMOKE TEST RESULTS')
print('='*55)
all_passed = True
for lib, status in results.items():
    print(f'  {lib:<22} {status}')
    if status.startswith('✗'):
        all_passed = False
print('='*55)
if all_passed:
    print('  ALL TESTS PASSED — ready for 01_data.ipynb')
else:
    print('  SOME TESTS FAILED — fix ✗ items before continuing')

[12:38:00] DEPRECATION WARNING: please use MorganGenerator
No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading modules with transformers dependency. No module named 'transformers.models.roberta.tokenization_roberta_fast'
cannot import name 'Chemberta' from 'deepchem.models.torch_models' (C:\Users\PC\anaconda3\envs\drug_discovery\lib\site-packages\deepchem\mo

SMOKE TEST RESULTS
  RDKit                  ✓  Aspirin MW=180.2, FP len=2048
  PyTDC                  ✗  No module named 'tdc'
  DeepChem               ✓  v2.8.0
  Transformers           ✓  v5.7.0
  PyTorch Geometric      ✓  v2.7.0
  Optuna                 ✓  v4.8.0
  UMAP                   ✓  Imported successfully
  CodeCarbon             ✓  Imported successfully
  SOME TESTS FAILED — fix ✗ items before continuing


## 6. Save Project Config

In [10]:
import json, os

CONFIG = {
    'gpu':          'RTX 3070',
    'vram_gb':      8,
    'use_amp':      True,
    'batch_size':   64,
    'num_workers':  4,
    'random_seed':  42,
    'test_size':    0.2,
    'val_size':     0.1,
    'target':       'EGFR',
    'tdc_dataset':  'EGFR_DrugBankFull',
    'task':         'regression',
    'models':       ['rf_morgan', 'attentivefp', 'mpnn_deepchem', 'chemberta2'],
    'data_dir':     './data/',
    'model_dir':    './models/',
    'results_dir':  './results/',
    'gpt_rosalind_bixbench': None,
}

os.makedirs('./data',    exist_ok=True)
os.makedirs('./models',  exist_ok=True)
os.makedirs('./results', exist_ok=True)
os.makedirs('./utils',   exist_ok=True)

with open('./config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print('config.json saved successfully.')
print()
print(json.dumps(CONFIG, indent=2))

config.json saved successfully.

{
  "gpu": "RTX 3070",
  "vram_gb": 8,
  "use_amp": true,
  "batch_size": 64,
  "num_workers": 4,
  "random_seed": 42,
  "test_size": 0.2,
  "val_size": 0.1,
  "target": "EGFR",
  "tdc_dataset": "EGFR_DrugBankFull",
  "task": "regression",
  "models": [
    "rf_morgan",
    "attentivefp",
    "mpnn_deepchem",
    "chemberta2"
  ],
  "data_dir": "./data/",
  "model_dir": "./models/",
  "results_dir": "./results/",
  "gpt_rosalind_bixbench": null
}


## 7. GPU Baseline Speed Test

In [11]:
import torch, time, json

if not torch.cuda.is_available():
    print('Skipping — no CUDA available')
else:
    device = torch.device('cuda')
    bench  = {}
    print('Running GPU baseline benchmark...')

    for dtype, label in [(torch.float32, 'FP32'), (torch.float16, 'FP16')]:
        for n in [512, 1024, 2048, 4096]:
            a = torch.randn(n, n, device=device, dtype=dtype)
            b = torch.randn(n, n, device=device, dtype=dtype)
            torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(10):
                c = a @ b
            torch.cuda.synchronize()
            elapsed = (time.time() - t0) / 10 * 1000
            bench[f'{label} {n}x{n}'] = f'{elapsed:.2f} ms'
            del a, b, c

    torch.cuda.empty_cache()

    print('FP32:')
    for k, v in bench.items():
        if 'FP32' in k:
            print(f'  {k:<25} {v}')
    print('FP16 (mixed precision):')
    for k, v in bench.items():
        if 'FP16' in k:
            print(f'  {k:<25} {v}')
    print()
    print('FP16 should be ~2x faster — confirms AMP is worth enabling.')

    with open('./results/gpu_baseline.json', 'w') as f:
        json.dump(bench, f, indent=2)
    print('Baseline saved to results/gpu_baseline.json')

Running GPU baseline benchmark...
FP32:
  FP32 512x512              0.30 ms
  FP32 1024x1024            2.00 ms
  FP32 2048x2048            6.40 ms
  FP32 4096x4096            15.00 ms
FP16 (mixed precision):
  FP16 512x512              3.60 ms
  FP16 1024x1024            0.10 ms
  FP16 2048x2048            0.30 ms
  FP16 4096x4096            2.40 ms

FP16 should be ~2x faster — confirms AMP is worth enabling.
Baseline saved to results/gpu_baseline.json


## ✓ Setup Complete

If all cells above ran without errors, your environment is fully configured.

**Next step:** Open `01_data.ipynb`

---
### Quick Reference — RTX 3070 Tips

| Problem | Solution |
|---|---|
| `CUDA out of memory` | Reduce `batch_size` in config.json from 64 → 32 |
| Training very slow | Check `USE_AMP=True` and `torch.backends.cudnn.benchmark=True` |
| High VRAM from previous run | `torch.cuda.empty_cache()` between models |
| DeepChem not finding GPU | Run `import deepchem as dc; dc.utils.get_default_device()` |

### Energy Tracking Quick Reference

```python
from codecarbon import EmissionsTracker
tracker = EmissionsTracker(project_name='model_name',
                           output_dir='./results/', log_level='warning')
tracker.start()
# ... your training code ...
kg_co2 = tracker.stop()
```